In [1]:
import os
from pathlib import Path

# IMPORTS DESDE MÓDULO DE CONFIGURACIÓN
from Configuracion import ANALITICOS_PARQUET, MODELOS_ML

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RUTA_FEATURES = ANALITICOS_PARQUET / "dataset_enriquecido_final.parquet"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ProyectoFinal_Guia08")
    .config("spark.driver.memory", "6g")
    .getOrCreate()
)

print("Spark:", spark.version)

Spark: 3.5.3


In [2]:
df = spark.read.parquet(str(RUTA_FEATURES.resolve()))
total = df.count()
print("Filas:", total, "| Columnas:", len(df.columns))

df = df.withColumn("es_ataque", F.when(F.col("label") == "BENIGN", 0).otherwise(1))

print("\nDistribución del target:")
(
    df.groupBy("es_ataque").count()
    .withColumn("porcentaje", F.round(F.col("count") / total * 100, 2))
    .show()
)

print("Detalle por tipo de tráfico:")
df.groupBy("label").count().orderBy(F.desc("count")).show(20, truncate=False)

Filas: 2827562 | Columnas: 93

Distribución del target:
+---------+-------+----------+
|es_ataque|  count|porcentaje|
+---------+-------+----------+
|        1| 556555|     19.68|
|        0|2271007|     80.32|
+---------+-------+----------+

Detalle por tipo de tráfico:
+--------------------------+-------+
|label                     |count  |
+--------------------------+-------+
|BENIGN                    |2271007|
|DoS Hulk                  |230123 |
|PortScan                  |158804 |
|DDoS                      |128025 |
|DoS GoldenEye             |10293  |
|FTP-Patator               |7935   |
|SSH-Patator               |5897   |
|DoS slowloris             |5796   |
|DoS Slowhttptest          |5499   |
|Bot                       |1956   |
|Web Attack - Brute Force  |1507   |
|Web Attack - XSS          |652    |
|Infiltration              |36     |
|Web Attack - Sql Injection|21     |
|Heartbleed                |11     |
+--------------------------+-------+



In [3]:
TARGET = "es_ataque"

COLS_NUMERICAS = [
    "flow_duration", "total_fwd_packets", "total_backward_packets",
    "total_length_of_fwd_packets", "total_length_of_bwd_packets",
    "fwd_packet_length_mean", "bwd_packet_length_mean",
    "packet_length_mean", "packet_length_std",
    "flow_bytes_s", "flow_packets_s",
    "flow_iat_mean", "fwd_iat_mean", "bwd_iat_mean",
    "down_up_ratio", "init_win_bytes_forward", "init_win_bytes_backward",
    "ratio_fwd_bwd_bytes", "total_flags_activadas",
]
COLS_CATEGORICAS = ["protocol", "categoria_duracion"]
COLS_MODELO = COLS_NUMERICAS + COLS_CATEGORICAS

print("Columnas que no existen en el dataset:", [c for c in COLS_MODELO if c not in df.columns])

FRACCION_MUESTRA = 0.20
df_muestra = df.sampleBy(TARGET, fractions={0: FRACCION_MUESTRA, 1: FRACCION_MUESTRA}, seed=42)
pdf = df_muestra.select(COLS_MODELO + [TARGET]).toPandas()

print("\nMuestra:", pdf.shape)
print(pdf[TARGET].value_counts(normalize=True).round(4))

nulos = pdf.isnull().sum()
print("\nNulos por columna:")
print(nulos[nulos > 0] if (nulos > 0).any() else "Sin nulos")

infinitos = np.isinf(pdf[COLS_NUMERICAS]).sum()
print("\nInfinitos por columna:")
print(infinitos[infinitos > 0] if (infinitos > 0).any() else "Sin infinitos")

Columnas que no existen en el dataset: []

Muestra: (566097, 22)
es_ataque
0    0.8027
1    0.1973
Name: proportion, dtype: float64

Nulos por columna:
Sin nulos

Infinitos por columna:
Sin infinitos


In [4]:
X = pdf[COLS_MODELO]
y = pdf[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

for nombre, yy in [("Entrenamiento", y_train), ("Validación", y_val), ("Prueba", y_test)]:
    print(f"{nombre}: {len(yy)} registros | % ataque: {yy.mean() * 100:.2f}")

Entrenamiento: 396267 registros | % ataque: 19.73
Validación: 84915 registros | % ataque: 19.73
Prueba: 84915 registros | % ataque: 19.73


In [5]:
preprocesador = ColumnTransformer([
    ("num", StandardScaler(), COLS_NUMERICAS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), COLS_CATEGORICAS),
])

modelo = Pipeline([
    ("prep", preprocesador),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

modelo.fit(X_train, y_train)

print("Modelo entrenado")
print("Iteraciones usadas:", modelo.named_steps["clf"].n_iter_[0])

Modelo entrenado
Iteraciones usadas: 179


In [6]:
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

metricas = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
}

print("Métricas en el conjunto de prueba:")
for nombre, valor in metricas.items():
    print(f"{nombre}: {valor:.4f}")

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("\nMatriz de confusión:")
print(f"TN = {tn} | FP = {fp}")
print(f"FN = {fn} | TP = {tp}")

Métricas en el conjunto de prueba:
Accuracy: 0.8510
Precision: 0.5732
Recall: 0.9571
F1-Score: 0.7170
ROC-AUC: 0.9573

Matriz de confusión:
TN = 56229 | FP = 11936
FN = 718 | TP = 16032


In [7]:
nombres = modelo.named_steps["prep"].get_feature_names_out()
coefs = modelo.named_steps["clf"].coef_[0]

df_coef = pd.DataFrame({"variable": nombres, "coeficiente": coefs})
df_coef["variable"] = df_coef["variable"].str.replace("num__", "").str.replace("cat__", "")
df_coef["abs"] = df_coef["coeficiente"].abs()
df_coef["importancia"] = pd.qcut(
    df_coef["abs"].rank(method="first"), 3, labels=["Baja", "Media", "Alta"]
)

df_coef = df_coef.sort_values("abs", ascending=False).reset_index(drop=True)
print(df_coef[["variable", "coeficiente", "importancia"]].round(4).to_string())

                       variable  coeficiente importancia
0        total_backward_packets     -37.5639        Alta
1             total_fwd_packets     -28.6357        Alta
2                   protocol_17     -27.5173        Alta
3                    protocol_6      18.7587        Alta
4      categoria_duracion_Medio      -3.2647        Alta
5      categoria_duracion_Largo      -2.9728        Alta
6        fwd_packet_length_mean      -2.6701        Alta
7         total_flags_activadas      -2.4805        Alta
8      categoria_duracion_Corto      -2.3257       Media
9                  fwd_iat_mean       1.9038       Media
10  total_length_of_bwd_packets      -1.3352       Media
11            packet_length_std       1.2586       Media
12  total_length_of_fwd_packets       0.8611       Media
13      init_win_bytes_backward      -0.7733       Media
14                flow_duration      -0.6299       Media
15           packet_length_mean      -0.4449       Media
16                down_up_ratio

In [8]:
pdf_modelado = pd.concat([
    X_train.assign(es_ataque=y_train, conjunto="train"),
    X_val.assign(es_ataque=y_val, conjunto="val"),
    X_test.assign(es_ataque=y_test, conjunto="test"),
])

RUTA_MODELADO = ANALITICOS_PARQUET / "dataset_modelado_guia08.parquet"
pdf_modelado.to_parquet(RUTA_MODELADO, index=False)

print(pdf_modelado.shape)
print(pdf_modelado["conjunto"].value_counts().to_dict())

(566097, 23)
{'train': 396267, 'val': 84915, 'test': 84915}


In [9]:
import pickle

# Ajusta esta ruta a la carpeta donde quieras guardarlo
archivo_destino = MODELOS_ML / "datos_split_guia8.pkl"

datos_para_andres = {
    "X_train": X_train,
    "X_val": X_val,
    "X_test": X_test,
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test,
    "COLS_NUMERICAS": COLS_NUMERICAS,
    "COLS_CATEGORICAS": COLS_CATEGORICAS,
    "TARGET": TARGET
}

with open(archivo_destino, "wb") as f:
    pickle.dump(datos_para_andres, f)

print(f"Archivo guardado en: {archivo_destino.resolve()}")

Archivo guardado en: C:\Users\galle\Desktop\IA Datos\Jupyter\Proyecto_GDIABD_Cybersecurity_Data_Analytics\Datos\Modelos\datos_split_guia8.pkl


In [10]:
print(pd.crosstab(pdf["protocol"], pdf["es_ataque"]))

es_ataque       0       1
protocol                 
0             346       2
6          254160  111664
17         199925       0


In [11]:
from sklearn.base import clone

modelo_sin_peso = Pipeline([
    ("prep", clone(preprocesador)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
modelo_sin_peso.fit(X_train, y_train)

def evaluar(m, X, y):
    pred = m.predict(X)
    prob = m.predict_proba(X)[:, 1]
    return {
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred),
        "Recall": recall_score(y, pred),
        "F1-Score": f1_score(y, pred),
        "ROC-AUC": roc_auc_score(y, prob),
    }

comparacion = pd.DataFrame({
    "balanced": evaluar(modelo, X_val, y_val),
    "sin_peso": evaluar(modelo_sin_peso, X_val, y_val),
}).round(4)

print(comparacion)

           balanced  sin_peso
Accuracy     0.8514    0.9245
Precision    0.5741    0.8832
Recall       0.9558    0.7115
F1-Score     0.7174    0.7881
ROC-AUC      0.9569    0.9558
